In [11]:

import gymnasium as gym
import torch
from pathlib import Path
from torch.distributions import Categorical
import torch.nn as nn

DEVICE = torch.device("cuda")


class ActorCriticNet(nn.Module):
    def __init__(self, obs_dim=8, n_actions=4):
        super().__init__()

        self.actor = nn.Sequential(
          nn.Linear(obs_dim, 128),
          nn.LayerNorm(128),
          nn.Tanh(),
          nn.Linear(128, 256),
          nn.LayerNorm(256),
          nn.Tanh(),
          nn.Linear(256, n_actions)
        )

        self.critic = nn.Sequential(
          nn.Linear(obs_dim, 128),
          nn.LayerNorm(128),
          nn.Tanh(),
          nn.Linear(128, 256),
          nn.LayerNorm(256),
          nn.Tanh(),
          nn.Linear(256, 1)
        )

    def forward(self, x):
        return self.actor(x), self.critic(x).squeeze(-1)

# 1. 저장된 모델 불러오기
model_path = Path("./ac_lunar_results/best_model-1.pth")

checkpoint = torch.load(
    model_path,
    map_location=DEVICE,
    weights_only=True
)

ac = ActorCriticNet(8, 4).to(DEVICE)
ac.load_state_dict(checkpoint["state_dict"])
ac.eval()

# 2. 게임 화면 생성
for seed in (42*i for i in range(1, 42)):
  print(f"seed {seed} test")
  env = gym.make("LunarLander-v3", render_mode="human")

  obs, info = env.reset(seed=seed)
  total_reward = 0

  # 3. 학습된 Actor로 게임 실행
  with torch.no_grad():
      for t in range(1500):

          state = torch.as_tensor(
              obs,
              dtype=torch.float32,
              device=DEVICE
          )

          logits, value = ac(state)

          # 가장 확률 높은 행동 선택
          # action = torch.argmax(logits, dim=-1).item()

          # 샘플링하기
          dist = Categorical(logits=logits)
          action = dist.sample().item()

          obs, reward, terminated, truncated, info = env.step(action)

          total_reward += float(reward)

          if terminated or truncated:
              break

  env.close()

  print(f"Total Reward: {total_reward:.2f}")
  print(f"Total Steps: {t + 1}")


seed 42 test
Total Reward: 230.98
Total Steps: 435
seed 84 test
Total Reward: -121.22
Total Steps: 1000
seed 126 test
Total Reward: 264.27
Total Steps: 374
seed 168 test
Total Reward: 276.02
Total Steps: 351
seed 210 test
Total Reward: 2.15
Total Steps: 167
seed 252 test
Total Reward: -25.84
Total Steps: 184
seed 294 test
Total Reward: 278.62
Total Steps: 397
seed 336 test
Total Reward: 209.02
Total Steps: 517
seed 378 test
Total Reward: 12.33
Total Steps: 229
seed 420 test
Total Reward: 25.31
Total Steps: 237
seed 462 test
Total Reward: -153.55
Total Steps: 1000
seed 504 test


KeyboardInterrupt: 

In [18]:
import torch
from pathlib import Path

pth = torch.load(Path(r"C:\LANG_CHAIN_2026\2026-05-19_KDT_lang_chain\01_class\2026_10_08_multimodal\ac_lunar_results\tuned_online_ac\best_model.pth"))
pth.keys()

dict_keys(['episode', 'evaluation_mean', 'evaluation_std', 'model_architecture', 'state_dict', 'model_config', 'optimizer_state_dict', 'scheduler_state_dict'])

In [8]:
from torch import nn
class ActorCriticNet(nn.Module):
    """Independent policy and value networks: no shared-gradient conflict."""

    def __init__(self, obs_dim=8, n_actions=4):
        super().__init__()

        self.actor = nn.Sequential(
            nn.Linear(obs_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 128),
            nn.Tanh(),
            nn.Linear(128, n_actions),
        )
        self.critic = nn.Sequential(
            nn.Linear(obs_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 128),
            nn.Tanh(),
            nn.Linear(128, 1),
        )

        # Small initial actor logits; independently initialized value head.
        for network in (self.actor, self.critic):
            for layer in network:
                if isinstance(layer, nn.Linear):
                    # 직교초기화
                    # 가중치 행렬의 행 또는 열이 서로 직교하도록 초기화하는 방식
                    # 이를 통해서 초기 신호와 gradient의 크기가 층을 통과하면서 지나치게 커지거나 작아지는 현상 완화
                    nn.init.orthogonal_(layer.weight, gain=nn.init.calculate_gain("tanh"))
                    nn.init.zeros_(layer.bias)
        nn.init.orthogonal_(self.actor[-1].weight, gain=0.01)
        nn.init.orthogonal_(self.critic[-1].weight, gain=1.0)

    def forward(self, x):
        return self.actor(x), self.critic(x).squeeze(-1)

In [9]:
ac

ActorCriticNet(
  (actor): Sequential(
    (0): Linear(in_features=8, out_features=128, bias=True)
    (1): Tanh()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): Tanh()
    (4): Linear(in_features=128, out_features=4, bias=True)
  )
  (critic): Sequential(
    (0): Linear(in_features=8, out_features=128, bias=True)
    (1): Tanh()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): Tanh()
    (4): Linear(in_features=128, out_features=1, bias=True)
  )
)

In [17]:
for k, v in pth["state_dict"].items():
  print(k, v.shape)

body.0.weight torch.Size([128, 8])
body.0.bias torch.Size([128])
body.1.weight torch.Size([128])
body.1.bias torch.Size([128])
body.3.weight torch.Size([128, 128])
body.3.bias torch.Size([128])
body.4.weight torch.Size([128])
body.4.bias torch.Size([128])
body.6.weight torch.Size([128, 128])
body.6.bias torch.Size([128])
actor.0.weight torch.Size([128, 128])
actor.0.bias torch.Size([128])
actor.1.weight torch.Size([128])
actor.1.bias torch.Size([128])
actor.3.weight torch.Size([256, 128])
actor.3.bias torch.Size([256])
actor.4.weight torch.Size([256])
actor.4.bias torch.Size([256])
actor.6.weight torch.Size([4, 256])
actor.6.bias torch.Size([4])
critic.0.weight torch.Size([128, 128])
critic.0.bias torch.Size([128])
critic.1.weight torch.Size([128])
critic.1.bias torch.Size([128])
critic.3.weight torch.Size([256, 128])
critic.3.bias torch.Size([256])
critic.4.weight torch.Size([256])
critic.4.bias torch.Size([256])
critic.6.weight torch.Size([1, 256])
critic.6.bias torch.Size([1])


In [24]:
import gymnasium as gym
from torch.distributions import Categorical

DEVICE = torch.device("cuda")

ac = ActorCriticNet().to(DEVICE)
ac.load_state_dict(pth["state_dict"])
ac.eval()


# 2. 게임 화면 생성
for seed in (42*i for i in range(1, 42)):
  env = gym.make("LunarLander-v3", render_mode="human")

  obs, info = env.reset(seed=seed)
  total_reward = 0

  # 3. 학습된 Actor로 게임 실행
  with torch.no_grad():
      for t in range(1500):

          state = torch.as_tensor(
              obs,
              dtype=torch.float32,
              device=DEVICE
          )

          logits, value = ac(state)

          # 가장 확률 높은 행동 선택
          # action = torch.argmax(logits, dim=-1).item()

          # 샘플링하기
          dist = Categorical(logits=logits)
          action = dist.sample().item()

          obs, reward, terminated, truncated, info = env.step(action)

          total_reward += float(reward)

          if terminated or truncated:
              break

  env.close()

  print(f"seed {seed:4d} | Total Reward: {total_reward:6.2f}\t| Total Steps: {t + 1:4d}")
 


seed   42| Total Reward: 113.60	| Total Steps: 1000
seed   84| Total Reward: 249.17	| Total Steps:  245
seed  126| Total Reward: 274.31	| Total Steps:  259
seed  168| Total Reward: 247.47	| Total Steps:  827
seed  210| Total Reward: 240.50	| Total Steps:  402
seed  252| Total Reward: -10.65	| Total Steps:  216
seed  294| Total Reward:  35.70	| Total Steps:  164
seed  336| Total Reward: 254.08	| Total Steps:  260
seed  378| Total Reward: 240.95	| Total Steps:  285
seed  420| Total Reward: 275.06	| Total Steps:  271
seed  462| Total Reward: 219.46	| Total Steps:  194
seed  504| Total Reward: 222.58	| Total Steps:  289
seed  546| Total Reward:  80.85	| Total Steps:  159
seed  588| Total Reward:  56.16	| Total Steps:  182
seed  630| Total Reward:  59.07	| Total Steps:  198
seed  672| Total Reward:  50.90	| Total Steps:  256
seed  714| Total Reward:   9.22	| Total Steps:  169


KeyboardInterrupt: 